[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paabL/rl-hvac-lab/blob/main/en/rl_lab_student_en.ipynb)

# Can one thermostat heat a thousand houses it has never seen?

**IFT3700/IFT6758 · Data Science · RL lab · student version (prototype)**

In Lecture 2, RL was used to align LLMs with human preferences. Today we ask it for **general heating control**: one controller, the same weights for every house, with no model and no tuning for each one. We try it on 1,000 houses in Montreal in January, climbing a ladder with five rungs.

> ⚠️ **Run the Setup cell first.** It downloads the companion files (`lab_utils_en.py`, `schemas.py`, `agents/`, the weather CSV) by itself. If it cannot reach GitHub, it shows a button to upload **`rl_lab_en.zip`**. Skipping this cell gives `No module named 'lab_utils_en'`.

> **Exercises.** Replace each `___` with your code, then run the cell. While an exercise is empty, the notebook keeps going with the lab's version.

> **Prototype.** The weather is **synthetic** for now. Agents are pre-trained on CPU (1,200 PPO iterations for the memory agents, about 20 minutes each, one random seed): the numbers come from this prototype, not from a final protocol.

## 0 · Setup

In [ ]:
# Session settings
SHOW_WIDGETS = True    # interactive sliders (False: static figures instead)
RUN_TRAINING = False   # True: retrains the "classic" agent live (~2 min on CPU)
REAL_WEATHER = False   # True: replaces the synthetic weather with Montreal Januaries 2015-2024 from Open-Meteo (needs Internet)

# Companion files: lab_utils_en.py, schemas.py, blanks.py, agents/, montreal_weather.csv
import os, sys, urllib.request, zipfile
if not os.path.exists("blanks.py"):
    bundle = "rl_lab_en.zip"
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/paabL/rl-hvac-lab/main/en/rl_lab_en.zip", bundle)
        print("Companion files downloaded from GitHub.")
    except OSError:                                 # GitHub unreachable: ask for the zip
        try:
            from google.colab import files          # only exists in Colab
        except ImportError:
            raise FileNotFoundError("blanks.py not found: open this notebook from the en/ folder of the lab.")
        print("Download failed. Click 'Choose files' below and pick rl_lab_en.zip.")
        zips = [name for name in files.upload() if name.endswith(".zip")]
        assert zips, "That was not a .zip file: please upload rl_lab_en.zip."
        bundle = zips[0]
    archive = zipfile.ZipFile(bundle)
    archive.extractall(".", [f for f in archive.namelist() if not f.endswith(".ipynb")])
sys.path.insert(0, ".")

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from lab_utils_en import *
from schemas import house_diagram, loop_diagram, agent_diagram
from blanks import ___, check
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
torch.set_num_threads(4)

if REAL_WEATHER:
    download_open_meteo().to_csv("montreal_weather.csv", index=False)
weather = load_weather()
print(f"{len(weather)} hours of January weather, from {weather.t_ext.min():.0f} to {weather.t_ext.max():.0f} °C")

## 1 · Why control heating?

- **The stake.** Buildings in operation emitted 9.9 Gt of CO₂ in 2024, 26% of energy-related CO₂ (UNEP, *Global Status Report* 2026).
- **The lever.** Heating, ventilation and air conditioning (HVAC) are 38% of building energy (González-Torres et al., 2022).
- **The margin.** Reviews report 13 to 28% savings from better control (Drgoňa et al., 2020).
- **The peak.** In Quebec, on winter mornings and evenings, everyone heats at the same time. Hydro-Québec offers peak-time rates to spread demand: heat *before* the peak rather than during it.

Model predictive control (MPC) does this job very well... provided there is a model of each building. Building and calibrating that model house by house is too costly for millions of homes. Hence our through-line: **a general controller**. Can RL learn a single controller that adapts to each house on its own?

### The five regimes of the ladder

We never change the agent, only the world it trains in. At each rung it has to get a little smarter.

| Regime | In one sentence | Training world | Section |
| --- | --- | --- | --- |
| Classic RL | learn to control **one** given house | a single house | 3 |
| Robust | find **one compromise**, the same for every house | houses drawn at random, no memory | 4 |
| Memorize or recognize | recognize a house **already seen** in training | a few houses, with memory | 6 |
| Learn in context | estimate a house **never seen** by watching it, without changing its weights | a new house every episode | 6 |
| Meta-RL | **keep learning** when the house changes along the way | houses that change during the episode | 7 |

**Meta-RL in plain words: learning to learn.** During training the agent sees so many houses, and houses that change, that it learns a *way* of adapting. Once deployed, its weights no longer move: its memory learns the new house hour after hour, and relearns it if it changes.

## 2 · The problem: heating without paying peak prices

> **RL vocabulary** (new in this course)
> - **State** $s_t$: what the agent observes at hour $t$ (indoor and outdoor temperature, price, time of day).
> - **Action** $a_t$: the heat-pump power, from 0 to 100% in steps of 25%.
> - **Reward** $r_t = -\text{cost}_t$, with $\text{cost}_t = \text{price}_t \times \text{kWh}_t + \lambda \times \text{discomfort}_t$ (degree-hours outside 20-23 °C, $\lambda$ = 1 \$ per K·h).
> - **Return** $G_t = \sum_{k \ge t} r_k$: everything left to gain until the end of the week. The agent maximizes the return, not the next reward: paying a little now (preheating) can save more later.
> - **Policy** $\pi(a \mid s)$: the decision rule, the thing we learn.
> - **Episode**: one week, 168 hourly decisions, in one house.

**The house** has two temperatures: the air $T_a$, which reacts within minutes, and the thermal mass $T_m$ (walls, floors, furniture), which reacts over hours and is **invisible** to the thermostat. Each follows a heat balance, (heat capacity) × (rate of change) = sum of heat flows:

$$C_a \frac{dT_a}{dt} = \underbrace{UA\,(T_\text{ext} - T_a)}_{\text{loss to outside}} + \underbrace{H_{am}\,(T_m - T_a)}_{\text{exchange with the mass}} + Q_\text{pump} + \underbrace{Q_\text{int}}_{\text{people, appliances}}, \qquad C_m \frac{dT_m}{dt} = H_{am}\,(T_a - T_m)$$

Three parameters $\xi$ change from house to house, and the agent never sees them: the **insulation** $UA$ (W/K, heat lost per degree of difference with outside; higher means leakier), the **thermal mass** $C_m$ (J/K, how much heat the walls store) and the **heat-pump size** $\mu$ ($\mu = 1$: at full power the pump just holds 21 °C when it is -25 °C outside).

**The price** is peak on weekdays 6-9 am and 4-8 pm: 0.50 \$/kWh against 0.07 off-peak. The trick to learn: warm the walls *before* the peak, then coast through it. The thermal mass works like a battery.

**The heat pump** moves heat rather than making it: 1 kWh of electricity delivers COP kWh of heat (coefficient of performance), about 2.3 at -10 °C, less when it is colder.

In [ ]:
loop_diagram("en"); plt.show()

In [ ]:
house_diagram("en"); plt.show()

In [ ]:
# EXERCISE 1/5 · One hour of the two-temperature model (Euler method, 6 substeps of 10 min)
def my_thermal_step(Ta, Tm, u, t_ext, xi, substeps=6):
    q_pump = u * xi["mu"] * xi["UA"] * (T_SETPOINT - T_DESIGN)     # heat delivered (W)
    dt = 3600.0 / substeps                                          # seconds
    for _ in range(substeps):
        # TO DO: heat balance of the air (loss, exchange with the mass, pump, internal gains), then of the mass
        # Available: xi["UA"], xi["Cm"], H_AM, C_AIR, Q_INT, q_pump, t_ext
        dTa = ___
        dTm = ___
        Ta, Tm = Ta + dt * dTa, Tm + dt * dTm
    return Ta, Tm, q_pump / cop(t_ext) / 1000.0                     # electricity used (kWh) = heat / COP

xi_check = sample_houses(5, rng=np.random.default_rng(0))
check("Exercise 1", lambda: all(np.allclose(m, s) for m, s in zip(my_thermal_step(20.0, 19.0, 0.5, -10.0, xi_check), thermal_step(20.0, 19.0, 0.5, -10.0, xi_check))))

Here are 64 random houses over one week, all heated by the same proportional thermostat set to 21 °C. Grey bands are peak hours.

In [ ]:
xi_test, starts_test = test_bench(weather, n=256)      # 256 test houses and weather weeks, the same for every agent
d_th = evaluate(thermostat, weather, xi_test, starts_test)
peak = is_peak(np.arange(H))
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(d_th["Ta"][:64], aspect="auto", cmap="coolwarm", vmin=17, vmax=26)
for t in np.where(np.diff(np.r_[0, peak]) > 0)[0]:
    ax.axvspan(t - .5, t + 2.5 if t % 24 == 6 else t + 3.5, color="k", alpha=.12, lw=0)
ax.set(xlabel="hour of the week", ylabel="house", title="The same thermostat in 64 houses: smooth in some, wild swings in others")
plt.colorbar(im, label="indoor temperature (°C)"); plt.show()
print(f"Mean cost: {cost_per_day(d_th).mean():.2f} $/day; from {cost_per_day(d_th).min():.2f} to {cost_per_day(d_th).max():.2f} depending on the house")

## 3 · Classic RL: a single house

> **Reminder: Lecture 2.** Logistic regression is trained by $w_j \leftarrow w_j + \eta \sum_i (y_i - \hat y_i)\, x^i_j$.

> **The policy gradient.** We look for the weights $w$ that maximize the mean return $J(w) = \mathbb{E}_{\pi_w}[G_0]$. The policy-gradient theorem gives
> $$\nabla_w J = \mathbb{E}\Big[\sum_t G_t\, \nabla_w \log \pi_w(a_t \mid s_t)\Big].$$
> Our policy is a logistic regression on 7 features $x_t$: $\pi_w(\text{heat} \mid s_t) = \sigma(w^\top x_t)$. Then $\nabla_w \log \pi_w(a_t \mid s_t) = \big(a_t - \sigma(w^\top x_t)\big)\, x_t$, hence the **REINFORCE** update:
> $$w_j \leftarrow w_j + \eta \sum_t G_t\, \big(a_t - \sigma(w^\top x_t)\big)\, x_{t,j}.$$
> This is the logistic regression of Lecture 2: the label is the action taken, and each example is weighted by the return that followed it. Actions followed by a good return become more likely.

> **The baseline.** We replace $G_t$ by $G_t - b_t$, with $b_t$ the mean return at the same hour across houses. On average the gradient does not change (because $\mathbb{E}[\nabla \log \pi] = 0$), but it gets much less noisy.

In [ ]:
# EXERCISE 2/5 · The REINFORCE update
def my_update(w, X, A, P, G, eta):
    # X: (hours, houses, 7) features; A: actions taken, 0 or 1; P: probabilities π(heat); G: normalized returns
    # TO DO: return the new weights (average over hours and houses)
    return ___

r = np.random.default_rng(0); trial = (np.zeros(7), r.normal(size=(4, 3, 7)), r.integers(0, 2, (4, 3)), r.uniform(size=(4, 3)), r.normal(size=(4, 3)), 0.1)
done = check("Exercise 2", lambda: np.allclose(my_update(*trial), reinforce_update(*trial)), fallback=True)
w, hist = reinforce(weather, iterations=100, eta=5.0, update=my_update if done else None, verbose=False)
names = ["bias", "Ta - 21", "T_ext / 10", "peak now", "peak soon", "sin hour", "cos hour"]
print(pd.Series(w, index=names).round(2).to_string())
plt.plot(hist); plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("cost ($/day)"); plt.title("REINFORCE learns to heat in a few seconds"); plt.show()

**Question:** read the weights as you would for a logistic regression. Why is the weight of "Ta - 21" strongly negative? And the weight of "peak now"? Why does "peak soon" stay close to 0?

### Policy gradient is only one family among several

| Family | What is learned | Examples | In this lab |
| --- | --- | --- | --- |
| Policy gradient | the policy $\pi$ directly | REINFORCE, PPO | REINFORCE here; PPO for every later agent |
| Actor-critic | $\pi$ and a value $V$ used as a baseline | A2C, PPO, SAC | our agents: an actor head and a critic head |
| Value-based | the value $Q(s, a)$ of each action, then take the best | Q-learning, DQN | — |
| Model-based | a model of the dynamics, then plan | MPC, Dyna, MuZero | MPC, played here by the oracle |
| Imitation | copy an expert | behavior cloning, DAgger | — |

> **PPO in one sentence.** A policy gradient with a critic and a safeguard: it reuses each batch of experience several times, but clips the update as soon as the new policy drifts too far from the old one (the ratio $\pi_{new}/\pi_{old}$ is kept within $[0.8, 1.2]$). That is what makes training stable.

An on/off agent is crude. The same principle with five power levels, a memory and the PPO algorithm gives the "classic" agent. PPO is a policy-gradient method like REINFORCE, with a safeguard that keeps each update small, which makes training stable. Trained on this single house, it learns to preheat before the peak.

In [ ]:
if RUN_TRAINING:
    classic_agent = Agent(memory=True)
    train(classic_agent, Houses(weather, 256, xi=lambda: nominal_house(256), seed=1), iterations=80)
else:
    classic_agent, _ = load_agent("classic")

xi_nom = nominal_house(256)
for name, pol in [("thermostat", thermostat), ("preheating thermostat", preheating_thermostat), ("classic agent", classic_agent)]:
    print(f"{name:28s} {cost_per_day(evaluate(pol, weather, xi_nom, starts_test)).mean():6.2f} $/day")

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, pol in [("thermostat", thermostat), ("classic agent", classic_agent)]:
    ax.plot(evaluate(pol, weather, xi_nom, starts_test)["Ta"][0, :72], label=name)
for t in np.where(is_peak(np.arange(72)) > 0)[0]:
    ax.axvspan(t - .5, t + .5, color="k", alpha=.1, lw=0)
ax.axhspan(*COMFORT_BAND, color="g", alpha=.08); ax.set(xlabel="hour", ylabel="°C", title="The agent warms up before the peak and coasts down during it"); ax.legend(); plt.show()

> **On a known house, RL is useless: MPC is enough.** If the model of the house is known (its parameters $\xi$), model predictive control directly optimizes the next hours from the weather and price forecasts, with no training at all. In our numbers, the **oracle**, an agent that is given $\xi$, plays that role. RL becomes interesting when we want **one controller for every house**, with no model of each.

> **Towards general control.** The classic agent knows a single house: it is not general at all. Let us see what it does elsewhere.

## 4 · Robust: one compromise for all houses

The classic agent knows a single house. To drive a thousand, we train on houses drawn at random: this is the **data augmentation** of Lectures 1 and 7, called *domain randomization* in RL. A memoryless agent (an MLP that only sees the current observation) cannot tell houses apart, so the best it can learn is one compromise that works reasonably everywhere.

In [ ]:
# EXERCISE 3/5 · Draw n houses: ξ = nominal × exp(U(-w, w)), w = width × log(sqrt(factor))
def my_houses(n, width=1.0, rng=np.random.default_rng(0)):
    # TO DO: a dictionary {parameter: n values}, using NOMINAL and FACTOR
    return ___

seed = lambda: np.random.default_rng(0)
done = check("Exercise 3", lambda: all(np.allclose(my_houses(5, rng=seed())[k], sample_houses(5, rng=seed())[k]) for k in NOMINAL), fallback=True)
xi_demo = (my_houses if done else sample_houses)(2000, rng=seed())
print({k: (round(v.min(), 2), round(v.max(), 2)) for k, v in xi_demo.items()})

In [ ]:
oracle, _ = load_agent("oracle")      # is told ξ and the mass temperature: the yardstick, not a real controller
robust, _ = load_agent("robust")      # no memory, trained on random houses
res = {}
res["thermostat"] = cost_per_day(d_th)
res["classic (one house)"] = cost_per_day(evaluate(classic_agent, weather, xi_test, starts_test))
res["robust (no memory)"] = cost_per_day(evaluate(robust, weather, xi_test, starts_test))
res["oracle (sees ξ)"] = cost_per_day(evaluate(oracle, weather, xi_test, starts_test, privileged=True))
for k, v in res.items():
    print(f"{k:26s} {v.mean():6.2f} $/day   gap to oracle {100 * (v.mean() / res['oracle (sees ξ)'].mean() - 1):+6.1f} %")

gap = 100 * (res["robust (no memory)"] / res["oracle (sees ξ)"] - 1)
plt.figure(figsize=(6, 4.5))
sc = plt.scatter(xi_test["UA"], xi_test["mu"], c=gap, cmap="magma_r", s=25)
plt.colorbar(sc, label="extra cost of robust vs oracle (%)")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("insulation UA (W/K, higher = leakier)"); plt.ylabel("heat-pump size μ")
plt.title("The price of not knowing: the same compromise\ncosts up to over +50% depending on the house"); plt.show()

> **Towards general control.** One controller for every house: done, at about +10% above the oracle. But it is blind to the house. To do better, it would have to recognize it.

## 5 · Does the agent listen to its memory?

> **Why a memory can recognize the house.** Heating, then watching how the temperature responds, is what engineers do to model a building: **system identification**. A leaky house cools fast when the heat is cut; a small heat pump warms slowly when pushed. With a few hours of history (power requested, temperatures), a simple regression would estimate $UA$, $C_m$ and $\mu$. A memory can do this computation implicitly, without being told what to look for... if training pushes it to.

So we give the agent a memory: a GRU, which you can read as an MLP that reads its own previous output, $h_t = f(W[x_t, h_{t-1}] + w_0)$. On top of it, three heads.

In [ ]:
agent_diagram("en"); plt.show()

> **Actor-critic.** The actor picks the action. The critic $V(h_t)$ predicts the expected return and serves as a baseline: the actor learns from the **advantage** $A_t = G_t - V(h_t)$, "better or worse than expected". It is the baseline of Section 3, but learned.

> **Self-predictive RL.** The reward says whether the action was good, not what the house is: a weak, noisy signal for learning to identify it. The third head adds a self-supervised task: predict next hour's temperature from the memory and the action. The target is free, it is the thermometer. To predict well, the memory must contain the house. Making the agent predict its own future (*self-predictive RL*) helps a lot when the state is hidden (Schwarzer et al., 2021; Ni et al., 2024).

A toy with two weights shows why RL alone struggles. The memory writes $m = c\,\xi + \varepsilon$ (what it stores about the house, plus noise), the action reads $a = b\,m$, and the ideal action is $\xi$:

$$J(b, c) = \sigma_\xi^2\,(bc - 1)^2 + b^2 \sigma_\varepsilon^2$$

Here $\sigma_\xi$ measures how different the houses are and $\sigma_\varepsilon$ the noise. At $(0, 0)$, the robust policy, reading brings nothing while the memory is empty, and writing brings nothing while nobody reads: the gradient is zero. $(0, 0)$ is a **saddle point**, and gradient descent can linger there.

In [ ]:
# EXERCISE 4/5 · The gradient of the toy
def my_grad_J(b, c, s_xi=1.0, s_eps=1.0):
    # TO DO: dJ/db, dJ/dc
    return ___, ___

done = check("Exercise 4", lambda: np.allclose(my_grad_J(.3, -.7), grad_J(.3, -.7)), fallback=True)
gradient = my_grad_J if done else grad_J
print("Gradient at the saddle (0, 0):", gradient(0.0, 0.0))

def landscape(s_xi=1.0, head=0.0):
    bs = np.linspace(-1.5, 1.5, 200); B, Cc = np.meshgrid(bs, bs)
    plt.figure(figsize=(5.5, 5)); plt.contourf(B, Cc, np.log(J(B, Cc, s_xi) + head * (1 - Cc) ** 2 + 1e-3), 30, cmap="Blues_r")
    rng = np.random.default_rng(1)
    for _ in range(20):
        b, c = rng.normal(0, 1e-3, 2)        # start very close to the saddle
        traj = [(b, c)]
        for _ in range(150):
            gb, gc = gradient(b, c, s_xi)
            gc += -2 * head * (1 - c)        # the prediction head: a gradient for writing that does not go through reading
            b, c = b - 0.03 * gb, c - 0.03 * gc
            traj.append((b, c))
        traj = np.array(traj); plt.plot(traj[:, 0], traj[:, 1], "w-", lw=1); plt.plot(*traj[-1], "wo", ms=4)
    plt.plot(0, 0, "r*", ms=14, label="saddle: robust policy"); plt.xlim(-1.5, 1.5); plt.ylim(-1.5, 1.5)
    plt.xlabel("b: read the memory"); plt.ylabel("c: write the house into memory")
    plt.title(f"σξ = {s_xi:.1f}, prediction head = {head:.1f}"); plt.legend(loc="lower left"); plt.show()

if SHOW_WIDGETS:
    import ipywidgets as W
    W.interact(landscape, s_xi=W.FloatSlider(1.0, min=0.3, max=2.0, step=0.1, description="spread σξ"),
               head=W.FloatSlider(0.0, min=0.0, max=1.0, step=0.1, description="head"))
else:
    landscape(1.0, 0.0); landscape(2.0, 0.0); landscape(1.0, 0.5)

**Question:** widening the box of houses ($\sigma_\xi$) deepens the valley. Does it make the runs leave the saddle faster? What changes when the prediction head is switched on?

Same experiment on the real agents: a GRU trained by PPO **without** a prediction head, and the same GRU **with** the head.

In [ ]:
saddle, _ = load_agent("saddle")
in_context, _ = load_agent("in_context")
rows = []
for name, ag in [("GRU, no head (saddle)", saddle), ("GRU + head", in_context)]:
    d = evaluate(ag, weather, xi_test, starts_test)
    res[name] = cost_per_day(d)
    rows.append(dict(agent=name, cost=res[name].mean(),
                     erase_memory=erase_effect(ag, weather, xi_test, starts_test),
                     swap_memory=swap_effect(ag, weather, xi_test, starts_test),
                     **{f"probe R2 h{t}": r for t, r in probe(d, (6, 24, 72, 167)).items()}))
tab = pd.DataFrame(rows).set_index("agent").round(2)
tab

Each column is an experiment on the agent's memory:
- **cost**: \$/day on the 256 test houses.
- **erase_memory**: wipe the memory at hour 72; extra cost over the next 24 hours (%).
- **swap_memory**: swap the memories of two houses at hour 72, so each agent "believes" it is in the other house; extra cost over the next 24 hours (%).
- **probe R2**: a scikit-learn Ridge regression reads $\xi$ from the memory at hours 6, 24, 72 and 167; $R^2$ on houses the probe never saw. It measures what the memory *knows*, not whether the policy *uses* it.

**Question:** which agent really *listens to* its memory? Which one knows something without using it?

## 6 · Memorize or recognize, then learn in context

> **Reminder: Lectures 3 and 4.** Overfitting means memorizing the training set instead of understanding it.

Same agent (GRU + head), three training worlds: 4 houses, 16 houses, or a new house every episode. We test it on its training houses and on houses it has never seen.

In [ ]:
rows = []
for name, k in [("memorize_4", 4), ("memorize_16", 16), ("in_context", None)]:
    ag, _ = load_agent(name)
    if k:
        xi_k = sample_houses(k, 1.0, np.random.default_rng(11))      # the same k houses it was trained on
        idx = np.arange(256) % k
        xi_train = {key: v[idx] for key, v in xi_k.items()}
    else:
        xi_train = sample_houses(256, 1.0, np.random.default_rng(99))
    ref_train = cost_per_day(evaluate(oracle, weather, xi_train, starts_test, privileged=True)).mean()
    c_train = cost_per_day(evaluate(ag, weather, xi_train, starts_test)).mean()
    c_test = res["GRU + head"].mean() if not k else cost_per_day(evaluate(ag, weather, xi_test, starts_test)).mean()
    rows.append(dict(world=f"{k} houses" if k else "always new",
                     **{"training houses": 100 * (c_train / ref_train - 1),
                        "new houses": 100 * (c_test / res["oracle (sees ξ)"].mean() - 1)}))
gen = pd.DataFrame(rows).set_index("world").round(1)
gen.plot.bar(rot=0, figsize=(8, 3.5), ylabel="gap to oracle (%)", title="Few houses: it memorizes. Many: it generalizes")
plt.show(); gen

**Learning in context.** The weights are now frozen: only the memory changes as the agent watches the house. Below, each dot is a test house, placed by the first two principal components of the agent's 64-number memory and colored by its insulation.

In [ ]:
from sklearn.decomposition import PCA
d = evaluate(in_context, weather, xi_test, starts_test)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, t in zip(axs, (2, 96)):
    p = PCA(2).fit_transform(d["m"][:, t])
    s = ax.scatter(p[:, 0], p[:, 1], c=np.log(xi_test["UA"]), cmap="viridis", s=15)
    ax.set(title=f"Memory at hour {t}", xlabel="PC1", ylabel="PC2")
plt.colorbar(s, ax=axs, label="log insulation UA"); fig.suptitle("Reading the memory: houses line up by insulation as the hours go by", y=1.04); plt.show()

> **Towards general control.** The agent identifies a house it has never seen within a few hours, without touching its weights, like an LLM picking up a task from the examples in its prompt. One case is left: the house that changes.

## 7 · Meta-RL: when the house changes mid-episode

So far a house stays the same for the whole week. Real houses change: a heat pump fails, a window is left open. An agent that identified the house once must keep re-identifying it.

Why that is hard: an estimate that averages **all** of the past reacts more and more slowly. After a jump, it takes as long to correct halfway as its age at the time of the jump: that is its **half-life**. An estimate that forgets (an exponentially weighted mean) keeps a constant half-life.

In [ ]:
# EXERCISE 5/5 · The half-life of an estimate after a jump
def half_life(t0, method, n=400):
    x = pd.Series(np.r_[np.zeros(t0), np.ones(n - t0)])        # the parameter jumps from 0 to 1 at hour t0
    # TO DO: cumulative mean or exponential mean of x (half-life 12 h), using pandas
    est = ___
    return int(np.argmax(est.values[t0:] >= 0.5))               # hours after the jump until the estimate passes 0.5

if check("Exercise 5", lambda: abs(half_life(96, "cumulative") - 96) <= 1 and abs(half_life(192, "exponential") - 12) <= 1) is not None:
    print(pd.DataFrame({m: [half_life(t0, m) for t0 in (12, 24, 48, 96, 192)] for m in ("cumulative", "exponential")},
                       index=pd.Index([12, 24, 48, 96, 192], name="age at the jump (h)")))

In the simulator, the drift strikes at a random hour: the heat pump loses half its power, or the insulation degrades by a factor of 1.8. We compare the agent trained without drift to the one trained with drift.

In [ ]:
drift_agent, _ = load_agent("drift")
oracle_d, _ = load_agent("oracle_drift")
env_d = Houses(weather, 256, drift=True, seed=123); env_d.reset(xi=xi_test, starts=starts_test)   # same jumps as in evaluate()
t_jump = env_d.t_jump
after = lambda d: np.mean([d["cost"][i, t_jump[i]:t_jump[i] + 48].sum() for i in range(len(t_jump))]) / 2
rows, traces = {}, {}
for name, ag, priv in [("GRU + head, trained without drift", in_context, False), ("GRU + head, trained with drift", drift_agent, False),
                       ("oracle (sees ξ live)", oracle_d, True)]:
    d = evaluate(ag, weather, xi_test, starts_test, privileged=priv, drift=True)
    rows[name] = (cost_per_day(d).mean(), after(d)); traces[name] = d["Ta"]
ref = rows["oracle (sees ξ live)"]
drift_tab = pd.DataFrame({k: {"cost over the week ($/day)": v[0], "cost in the 48 h after the jump ($/day)": v[1],
                              "gap to oracle after the jump (%)": 100 * (v[1] / ref[1] - 1)} for k, v in rows.items()}).T.round(2)
drift_tab

In [ ]:
# one house whose heat pump loses half its power, picked where the gap between the two agents is clearest
names = list(traces)
disc_after = lambda Ta, i: np.maximum(COMFORT_BAND[0] - Ta[i, t_jump[i]:t_jump[i] + 48], 0).sum()
candidates = [j for j in np.where(env_d.jump_type == 0)[0] if disc_after(traces[names[2]], j) < 2]
i = max(candidates, key=lambda j: disc_after(traces[names[0]], j) - disc_after(traces[names[1]], j))

def gap_half_life(Ta, Ta_ref, t0, smooth=6):
    """Hours after the jump until the temperature gap to the oracle falls below half its maximum."""
    g = pd.Series(Ta_ref - Ta).rolling(smooth, min_periods=1).mean().values[t0:]
    top = int(g.argmax())
    below = np.where(g[top:] < g[top] / 2)[0]
    return top + int(below[0]) if len(below) else len(g), g[top]

fig, ax = plt.subplots(figsize=(11, 4))
for name, Ta in traces.items():
    ax.plot(Ta[i], label=name, lw=1.2)
t0 = t_jump[i]
ax.axvline(t0, color="r", ls="--", label="heat pump loses 50%"); ax.axhspan(*COMFORT_BAND, color="g", alpha=.08)
low = min(Ta[i].min() for Ta in traces.values())
ax.set_ylim(low - 1.3, None)
for k, (name, y) in enumerate([(names[0], low - 0.45), (names[1], low - 0.95)]):
    n12, top = gap_half_life(traces[name][i], traces[names[2]][i], t0)
    if top > 0.2:
        ax.annotate("", xy=(t0 + n12, y), xytext=(t0, y), arrowprops=dict(arrowstyle="<->", color=f"C{k}", lw=1.5))
        ax.text(t0 + n12 + 2, y, f"half-life ≈ {n12} h", va="center", color=f"C{k}", fontsize=10)
    else:
        ax.text(t0 + 2, y, "negligible gap to the oracle", va="center", color=f"C{k}", fontsize=10)
ax.set(xlabel="hour", ylabel="°C", title="After the jump: who adapts, and how fast?"); ax.legend(fontsize=9, loc="upper right"); plt.show()

The **half-life** can be read on the figure: the time after the jump for the temperature gap to the oracle to fall back to half its maximum.

**Question:** compare with Exercise 5. Which agent behaves like the cumulative mean, and which like the exponential one?

## 8 · Wrap-up: where each agent stopped

In [ ]:
summary = pd.DataFrame({"cost ($/day)": {k: v.mean() for k, v in res.items()}})
summary["gap to oracle (%)"] = 100 * (summary["cost ($/day)"] / res["oracle (sees ξ)"].mean() - 1)
summary.sort_values("cost ($/day)", ascending=False).round(2)

| Regime | Training world | Agent of the day | What it can do |
| --- | --- | --- | --- |
| Classic RL | one house | classic agent | beats the thermostat at home, fails elsewhere |
| Robust | random houses | memoryless MLP; GRU without head, near the saddle | one compromise at +10% above the oracle |
| Memorize or recognize | 4 or 16 houses | GRU + head | with 4 houses it memorizes; from 16, it recognizes |
| Learn in context | a new house every episode | GRU + head | identifies a house never seen, at +3% above the oracle |
| Meta-RL | the house changes during the episode | GRU + head, trained with drift | re-adapts after a failure |

**General control, in short.** An MPC needs the model of each house; our last agent needs only its history. At full scale, the NOMAD research project behind this lab stayed near the saddle: the memory knew a little, but the policy did not read it. Making this work on real buildings is still an open question.